In [2]:
import numpy as np
import pandas as pd
from category_encoders import TargetEncoder
from lightgbm import LGBMRegressor
from pytabkit import RealMLP_TD_Regressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

g:\Interview Prep\Projeenv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [124]:
df = pd.read_csv("/kaggle/input/datasets/madankhatri123h/dataset2/train-test.csv")
df.sample(5)

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
22124,TR-022125,Nashville,Greensboro,35.29479,-88.08915,35.98195,-82.78379,372.9,Reefer,27039.0,2025-05-20,1.28223,1.47590,1001.74
21474,TR-021475,Charleston,Hartford,34.15334,-79.74475,39.55328,-72.18051,661.7,Dry Van,35549.0,2025-05-16,1.35979,1.93779,1443.50
2613,TR-002614,Amarillo,Columbia,33.17137,-104.36789,34.63570,-83.28506,1402.0,Dry Van,31584.0,2025-01-17,0.96101,1.94906,2732.74
29973,TR-029974,Atlanta,Tampa,34.84933,-86.28940,28.35765,-86.94782,532.4,Flatbed,28636.0,2025-07-08,1.20714,1.61527,1333.38
25556,TR-025557,Shreveport,Richmond,32.54628,-92.31888,38.09122,-76.78906,1129.2,Dry Van,22461.0,2025-06-10,1.22033,1.95857,2219.24


In [125]:
df = df.drop(columns=['load_id'])

In [126]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   pickup        48000 non-null  object 
 1   delivery      48000 non-null  object 
 2   pickup_lat    48000 non-null  float64
 3   pickup_lon    48000 non-null  float64
 4   delivery_lat  48000 non-null  float64
 5   delivery_lon  48000 non-null  float64
 6   distance      48000 non-null  float64
 7   equipment     48000 non-null  object 
 8   weight        47700 non-null  float64
 9   date          48000 non-null  object 
 10  market_index  47626 non-null  float64
 11  quote_signal  48000 non-null  float64
 12  posted_rate   48000 non-null  float64
dtypes: float64(9), object(4)
memory usage: 4.8+ MB


In [127]:
# Returns a list of column names
catColumns = df.select_dtypes(include=['object']).columns.tolist()

In [128]:
for col in catColumns:
    print(f"----No of unique category {df[col].value_counts()}\n------")

----No of unique category pickup
Oklahoma City    1242
Lexington        1209
Bakersfield      1193
Fort Wayne       1170
Hartford         1150
                 ... 
Detroit           337
Washington        337
St. Louis         318
Las Vegas         277
Dallas            276
Name: count, Length: 64, dtype: int64
------
----No of unique category delivery
Lexington      1197
Fort Wayne     1176
Baton Rouge    1167
Bakersfield    1156
Hartford       1143
               ... 
St. Louis       344
Birmingham      318
Detroit         307
Dallas          306
Las Vegas       292
Name: count, Length: 64, dtype: int64
------
----No of unique category equipment
Dry Van    27202
Reefer     12045
Flatbed     8753
Name: count, dtype: int64
------
----No of unique category date
2025-06-17    198
2025-07-12    191
2025-05-30    191
2025-01-08    189
2025-03-31    188
             ... 
2025-08-19    132
2025-10-28    131
2025-08-11    127
2025-10-24    125
2025-02-18    115
Name: count, Length: 304, dtype

In [129]:
X = df.drop(columns=['posted_rate'])
y = df['posted_rate']

In [130]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [131]:
cols_with_nan = ['weight', 'market_index']

mean_imputer = SimpleImputer(strategy='mean')

X_train[cols_with_nan] = mean_imputer.fit_transform(X_train[cols_with_nan])
X_test[cols_with_nan] = mean_imputer.transform(X_test[cols_with_nan])

Creating new features based on these baseline coefficients so the tree-based model can bypass simple linear trends and focus entirely on capturing complex, non-linear relationships.

Training a baseline linear regression model on continuous and encoded features to extract the linear trends.

In [132]:
X_train['date']=X_train['date'].astype("datetime64[ns]")
X_test['date']=X_test['date'].astype("datetime64[ns]")

In [133]:
X_train['Hour']=X_train['date'].dt.hour
X_train['Month']=X_train['date'].dt.month
X_train['Dayofweek']=X_train['date'].dt.dayofweek
X_test['Hour']=X_test['date'].dt.hour
X_test['Month']=X_test['date'].dt.month
X_test['Dayofweek']=X_test['date'].dt.dayofweek

In [134]:
time_cycles = {
    'Hour': 24,       # Captures daily peaks and hourly shifts
    'Dayofweek': 7,   # Captures weekly fluctuations (weekdays vs. weekends)
    'Month': 12       # Captures long-term annual seasonality patterns
}

# Apply the trigonometric feature engineering loop to encode cycles smoothly
for col, p in time_cycles.items():
    X_train[f'_{col}_sin'] = np.sin(2 * np.pi * X_train[col] / p).astype('float32')
    X_train[f'_{col}_cos'] = np.cos(2 * np.pi * X_train[col] / p).astype('float32')
    X_test[f'_{col}_sin'] = np.sin(2 * np.pi * X_test[col] / p).astype('float32')
    X_test[f'_{col}_cos'] = np.cos(2 * np.pi * X_test[col] / p).astype('float32')

# Drop the raw time components and the original Datetime string so they don't cause redundancy
X_train = X_train.drop(columns=['date', 'Hour', 'Dayofweek', 'Month'])
X_test = X_test.drop(columns=['date', 'Hour', 'Dayofweek', 'Month'])

In [135]:
for c in ['weight','distance']:
    X_train[f'{c}_sq'] = X_train[c] ** 2
    X_test[f'{c}_sq'] = X_test[c] ** 2
    X_train[f'{c}_2'] = X_train[c].copy()
    X_test[f'{c}_2'] = X_test[c].copy()

In [136]:
# X_train.to_csv("mad.csv")

In [137]:
# Generate high-order cross-features by combining structural categories and discrete 
# binned numerical indicators to capture joint regional logistics constraints.

# Define a dictionary of the strategic combinations 
custom_combinations = {
    # 1. Truck Type + Destination Hub Strategy
    'equipment_delivery_hub': ['equipment', 'delivery'],
    }

for name, cols in custom_combinations.items():
    # 1. Initialize string concatenation on the first column in the recipe split
    X_train[name] = X_train[cols[0]].astype(str)
    X_test[name]  = X_test[cols[0]].astype(str)
    
    # 2. Loop through the remaining features to glue the strings together with an underscore
    for col in cols[1:]:
        X_train[name] = X_train[name] + '_' + X_train[col].astype(str)
        X_test[name]  = X_test[name]  + '_' + X_test[col].astype(str)
        
    # 3. Apply uniform .factorize() alignment across splits to avoid structural leaks
    combined_series = pd.concat([X_train[name], X_test[name]], ignore_index=True)
    encoded_integers, _ = combined_series.factorize()
    
    # 4. Reassign clean categorical integers back to the primary DataFrames as memory-efficient types
    X_train[name] = encoded_integers[:len(X_train)].astype('int16')
    X_test[name]  = encoded_integers[len(X_train):].astype('int16')

print("All high-order feature combinations successfully built and factorized!")


All high-order feature combinations successfully built and factorized!


In [3]:
class TargetEncoder_(BaseEstimator, TransformerMixin):
    """
    Target Encoder that supports multiple aggregation functions,
    internal cross-validation for leakage prevention, and smoothing.

    Parameters
    ----------
    cols_to_encode : list of str
        List of column names to be target encoded.

    aggs : list of str, default=['mean']
        List of aggregation functions to apply. Any function accepted by
        pandas' `.agg()` method is supported, such as:
        'mean', 'std', 'var', 'min', 'max', 'skew', 'nunique',
        'count', 'sum', 'median'.
        Smoothing is applied only to the 'mean' aggregation.

    cv : int, default=5
        Number of folds for cross-validation in fit_transform.

    smooth : float or 'auto', default='auto'
        The smoothing parameter `m`. A larger value puts more weight on the
        global mean. If 'auto', an empirical Bayes estimate is used.

    drop_original : bool, default=False
        If True, the original columns to be encoded are dropped.
    """
    def __init__(self, cols_to_encode, aggs=['mean'], cv=5, smooth='auto', drop_original=False):
        self.cols_to_encode = cols_to_encode
        self.aggs = aggs
        self.cv = cv
        self.smooth = smooth
        self.drop_original = drop_original
        self.mappings_ = {}
        self.global_stats_ = {}

    def fit(self, X, y):
        """Learn smoothed mappings from the complete training data."""
        temp_df = X.copy()
        temp_df['target'] = y

        for agg_func in self.aggs:
            self.global_stats_[agg_func] = y.agg(agg_func)

        for col in self.cols_to_encode:
            self.mappings_[col] = {}
            for agg_func in self.aggs:
                mapping = temp_df.groupby(col)['target'].agg(agg_func)

                if agg_func == 'mean':
                    counts = temp_df.groupby(col)['target'].count()
                    m = self.smooth
                    if self.smooth == 'auto':
                        variance_between = mapping.var()
                        avg_variance_within = temp_df.groupby(col)['target'].var().mean()
                        m = (
                            avg_variance_within / variance_between
                            if variance_between > 0
                            else 0
                        )
                    mapping = (
                        counts * mapping + m * self.global_stats_[agg_func]
                    ) / (counts + m)

                self.mappings_[col][agg_func] = mapping

        return self

    def transform(self, X):
        """Apply fitted mappings and use global statistics for unseen values."""
        X_transformed = X.copy()
        for col in self.cols_to_encode:
            for agg_func in self.aggs:
                new_col_name = f'TE_{col}_{agg_func}'
                map_series = self.mappings_[col][agg_func]
                X_transformed[new_col_name] = X[col].map(map_series)
                X_transformed[new_col_name].fillna(self.global_stats_[agg_func], inplace=True)

        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)

        return X_transformed

    def fit_transform(self, X, y):
        """Fit and transform the data with cross-validation to prevent leakage."""
        self.fit(X, y)
        encoded_features = pd.DataFrame(index=X.index)
        kf = KFold(n_splits=self.cv, shuffle=True, random_state=42)

        for train_idx, val_idx in kf.split(X, y):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_val = X.iloc[val_idx]
            temp_df_train = X_train.copy()
            temp_df_train['target'] = y_train

            for col in self.cols_to_encode:
                for agg_func in self.aggs:
                    new_col_name = f'TE_{col}_{agg_func}'
                    fold_global_stat = y_train.agg(agg_func)
                    mapping = temp_df_train.groupby(col)['target'].agg(agg_func)

                    if agg_func == 'mean':
                        counts = temp_df_train.groupby(col)['target'].count()
                        m = self.smooth
                        if self.smooth == 'auto':
                            variance_between = mapping.var()
                            avg_variance_within = temp_df_train.groupby(col)['target'].var().mean()
                            m = (
                                avg_variance_within / variance_between
                                if variance_between > 0
                                else 0
                            )
                        mapping = (counts * mapping + m * fold_global_stat) / (counts + m)

                    encoded_features.loc[X_val.index, new_col_name] = (
                        X_val[col].map(mapping).fillna(fold_global_stat)
                    )

        X_transformed = X.copy()
        for col in encoded_features.columns:
            X_transformed[col] = encoded_features[col]

        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)

        return X_transformed

In [139]:
# linear_input_features = [
#     'distance', 'weight', 'market_index', 'quote_signal',
#     'pickup', 'delivery',                   # Pre-calculated Target Encoded means
#     'equipment_Flatbed', 'equipment_Reefer'  # One-Hot Encoded dummy columns
# ]

# lr = LinearRegression()
# lr.fit(X_train[linear_input_features], y_train)

# print(f"--- Extracted Formula Baseline ---")
# print(f"Intercept (Base Value): {lr.intercept_:.4f}")
# for col, coef in zip(linear_input_features, lr.coef_):
#     print(f"{col} Linear Coefficient (Weight): {coef:.4f}")


print('''
outputs
--- Extracted Formula Baseline ---
Intercept (Base Value): -447.4607
distance Linear Coefficient (Weight): 1.8696
weight Linear Coefficient (Weight): 0.0073
market_index Linear Coefficient (Weight): 329.5113
quote_signal Linear Coefficient (Weight): 43.6723
pickup Linear Coefficient (Weight): -0.0182
delivery Linear Coefficient (Weight): -0.0168
equipment_Flatbed Linear Coefficient (Weight): 177.9682
equipment_Reefer Linear Coefficient (Weight): 284.7715
''')


outputs
--- Extracted Formula Baseline ---
Intercept (Base Value): -447.4607
distance Linear Coefficient (Weight): 1.8696
weight Linear Coefficient (Weight): 0.0073
market_index Linear Coefficient (Weight): 329.5113
quote_signal Linear Coefficient (Weight): 43.6723
pickup Linear Coefficient (Weight): -0.0182
delivery Linear Coefficient (Weight): -0.0168
equipment_Flatbed Linear Coefficient (Weight): 177.9682
equipment_Reefer Linear Coefficient (Weight): 284.7715



In [140]:
# Create the linear baseline anchor feature using only the numeric columns available at this stage
X_train['feature_formula'] = (
    (1.8696 * X_train['distance']) +
    (0.0073 * X_train['weight']) +
    (329.5113 * X_train['market_index']) +
    (43.6723 * X_train['quote_signal']) +
    (177.9682 * (X_train['equipment'] == 'Flatbed').astype(float)) +
    (284.7715 * (X_train['equipment'] == 'Reefer').astype(float)) +  
    (-447.4607) 
)

X_test['feature_formula'] = (
    (1.8696 * X_test['distance']) +
    (0.0073 * X_test['weight']) +
    (329.5113 * X_test['market_index']) +
    (43.6723 * X_test['quote_signal']) +
    (177.9682 * (X_test['equipment'] == 'Flatbed').astype(float)) +
    (284.7715 * (X_test['equipment'] == 'Reefer').astype(float)) +
    (-447.4607)
)


In [144]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 38400 entries, 596 to 15795
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   pickup                  38400 non-null  object 
 1   delivery                38400 non-null  object 
 2   pickup_lat              38400 non-null  float64
 3   pickup_lon              38400 non-null  float64
 4   delivery_lat            38400 non-null  float64
 5   delivery_lon            38400 non-null  float64
 6   distance                38400 non-null  float64
 7   equipment               38400 non-null  object 
 8   weight                  38400 non-null  float64
 9   market_index            38400 non-null  float64
 10  quote_signal            38400 non-null  float64
 11  _Hour_sin               38400 non-null  float32
 12  _Hour_cos               38400 non-null  float32
 13  _Dayofweek_sin          38400 non-null  float32
 14  _Dayofweek_cos          38400 non-null  f

In [150]:
cat_cols = ['pickup', 'delivery', 'equipment','equipment_delivery_hub'] 

te = TargetEncoder_(cols_to_encode=cat_cols, drop_original=True)
X_train_encoded = te.fit_transform(X_train, y_train)
X_test_encoded = te.transform(X_test)


print("\n2. Starting Optuna Hyperparameter Tuning...")

def objective_xgb(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    model = XGBRegressor(**params, n_estimators=100, random_state=42, n_jobs=-1)
    scores = cross_val_score(model, X_train_encoded, y_train, cv=3, scoring='neg_root_mean_squared_error')
    return scores.mean()

def objective_lgb(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
    }
    model = LGBMRegressor(**params, n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
    scores = cross_val_score(model, X_train_encoded, y_train, cv=3, scoring='neg_root_mean_squared_error')
    return scores.mean()

# Run tuning
optuna.logging.set_verbosity(optuna.logging.WARNING) 

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=15, show_progress_bar=True)
best_xgb_params = study_xgb.best_params

study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=15, show_progress_bar=True)
best_lgb_params = study_lgb.best_params


print("\n3. Building Stacking Ensemble...")
kf_stack = KFold(n_splits=5, shuffle=True, random_state=42)

def get_oof_predictions(model, X_data, y_data):
    oof_preds = np.zeros(len(X_data))
    X_data = X_data.reset_index(drop=True)
    y_data = y_data.reset_index(drop=True)
    
    for train_idx, val_idx in kf_stack.split(X_data, y_data):
        X_tr, y_tr = X_data.iloc[train_idx], y_data.iloc[train_idx]
        X_va = X_data.iloc[val_idx]
        
        model.fit(X_tr, y_tr)
        oof_preds[val_idx] = model.predict(X_va)
    return oof_preds

base_models = {
    'XGBoost': XGBRegressor(**best_xgb_params, n_estimators=250, random_state=42, n_jobs=-1),
    'LightGBM': LGBMRegressor(**best_lgb_params, n_estimators=250, random_state=42, n_jobs=-1, verbose=-1),
    'Linear': LinearRegression(),
    'RealMLP': RealMLP_TD_Regressor(device='cuda') 
}

oof_matrix = pd.DataFrame(index=X_train_encoded.index)
test_predictions_matrix = pd.DataFrame(index=X_test_encoded.index)

for name, model in base_models.items():
    print(f"Training {name} (Generating OOF & Test Predicts)...")
    
    oof_matrix[name] = get_oof_predictions(model, X_train_encoded, y_train)
    
    model.fit(X_train_encoded, y_train)
    test_predictions_matrix[name] = model.predict(X_test_encoded)

# Fit Meta-Model (Ridge) on the OOF predictions
print("\n4. Training Ridge Meta-Model...")
meta_model = RidgeCV(alphas=[0.1, 1.0, 10.0])
meta_model.fit(oof_matrix, y_train)

print("\n--- Meta-Model Weights ---")
for name, weight in zip(base_models.keys(), meta_model.coef_):
    print(f"{name}: {weight:.4f}")

print("\n5. Generating Final Test Set Predictions...")

final_test_preds = meta_model.predict(test_predictions_matrix)

# Evaluate final score
final_rmse = root_mean_squared_error(y_test, final_test_preds)
print(f"\n==========================================")
print(f" FINAL ENSEMBLE TEST RMSE: {final_rmse:.4f}")
print(f"==========================================")


2. Starting Optuna Hyperparameter Tuning...


/tmp/ipykernel_58/2220365290.py:69: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_transformed[new_col_name].fillna(self.global_stats_[agg_func], inplace=True)
/tmp/ipykernel_58/2220365290.py:69: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]


3. Building Stacking Ensemble...
Training XGBoost (Generating OOF & Test Predicts)...
Training LightGBM (Generating OOF & Test Predicts)...
Training Linear (Generating OOF & Test Predicts)...
Training RealMLP (Generating OOF & Test Predicts)...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=256` reached.
/usr/lo


4. Training Ridge Meta-Model...

--- Meta-Model Weights ---
XGBoost: 0.0760
LightGBM: 0.1834
Linear: 0.1177
RealMLP: 0.6255

5. Generating Final Test Set Predictions...

 FINAL ENSEMBLE TEST RMSE: 526.4839


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


In [151]:
import joblib

# 1. Package the entire fitted stack into a single dictionary
ensemble_pipeline = {
    'target_encoder': te,
    'base_models': base_models,
    'meta_model': meta_model
}

# 2. Save the dictionary to disk
joblib.dump(ensemble_pipeline, 'stacking_ensemble_pipeline.pkl')
print("💾 Stacking ensemble pipeline saved successfully as 'stacking_ensemble_pipeline.pkl'")

💾 Stacking ensemble pipeline saved successfully as 'stacking_ensemble_pipeline.pkl'


In [152]:
# Baseline Model
from sklearn.metrics import root_mean_squared_error


df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

features = [
    'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 
    'delivery_lat', 'delivery_lon', 'distance', 'weight', 
    'market_index', 'quote_signal', 'equipment', 'year', 'month', 'day'
]
target = 'posted_rate'

X = df[features].copy()
y = df[target]

cat_cols = ['pickup', 'delivery', 'equipment']
for col in cat_cols:
    if col in X.columns:
        X[col] = X[col].astype('category')

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    enable_categorical=True,
    tree_method='hist'
)

model.fit(X_train, y_train)

preds = model.predict(X_val)
rmse = root_mean_squared_error(y_val, preds)
print(f"🎯 Baseline Validation RMSE: {rmse:.4f}")

🎯 Baseline Validation RMSE: 558.5001
